In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Phishing Email Detection with AI and Cyber Threat Intelligence

Project Overview

This project aims to develop an AI-powered phishing email detection system by combining Cyber Threat Intelligence (CTI) with machine learning and Natural Language Processing (NLP). Phishing emails are a major cybersecurity threat, often used to steal sensitive information like login credentials or financial details. By leveraging CTI datasets and AI techniques, this system will classify emails as "phishing" or "legitimate" based on their content and metadata, providing a practical solution for enhancing email security.

The project uses a Kaggle-like dataset (inspired by real-world phishing datasets) and incorporates CTI principles, such as analyzing known phishing patterns and threat indicators, to improve detection accuracy. It’s beginner-friendly, focusing on accessible tools like Python, scikit-learn, and pre-trained NLP models like BERT.



Dataset Description

For this project, we’ll assume a dataset similar to those found on Kaggle, such as the "Enron Email Dataset" or a synthetic phishing email dataset. The dataset includes:





Features:





email_content: The body of the email (text).



subject: The email subject line.



sender: The sender’s email address or domain.



urls: Any URLs embedded in the email (if present).



metadata: Additional features like timestamp or email headers.



label: Binary label (1 for phishing, 0 for legitimate).



Size: Approximately 10,000 emails (70% legitimate, 30% phishing).



Source: Inspired by Kaggle datasets like "Spam Email Classification" or CTI repositories like the MITRE ATT&CK framework for phishing patterns.

Note: You can use a publicly available dataset from Kaggle (e.g., Spam/Ham Dataset) or create a synthetic dataset by combining legitimate emails with phishing samples from CTI sources like PhishTank.



Project Objectives





Data Preprocessing: Clean and preprocess email text, extract features, and incorporate CTI-based indicators (e.g., suspicious URLs, known phishing keywords).



Exploratory Data Analysis (EDA): Analyze email characteristics to identify patterns distinguishing phishing from legitimate emails.



Model Building: Train machine learning models (e.g., Logistic Regression, Random Forest) and a pre-trained NLP model (BERT) for email classification.



CTI Integration: Enhance the model with CTI insights, such as known phishing domains or tactics from the MITRE ATT&CK framework.



Evaluation: Assess model performance using metrics like accuracy, precision, recall, and F1-score.



Deployment: Provide a simple script for real-time email scanning.



Tools and Libraries





Python: Core programming language.



Pandas/NumPy: Data manipulation and analysis.



Scikit-learn: Machine learning models and evaluation metrics.



Transformers (Hugging Face): Pre-trained BERT model for NLP.



NLTK/Spacy: Text preprocessing and tokenization.



Matplotlib/Seaborn: Data visualization for EDA.



TLDExtract: Extract domains from URLs for CTI analysis.



Step-by-Step Implementation

1. Data Loading and Preprocessing





Load the dataset using Pandas.



Clean the email content: remove HTML tags, normalize text, and handle missing values.



Extract CTI-based features:





Check for suspicious keywords (e.g., "urgent," "login," "verify") from CTI phishing reports.



Identify domains in URLs and cross-reference with known phishing domains (e.g., PhishTank or MITRE ATT&CK).



Tokenize text for NLP models using NLTK or Hugging Face’s tokenizer.
Code Example:

In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from tldextract import extract
import nltk

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load dataset
data = pd.read_csv('phishing_email_dataset.csv')

# Clean email content
def clean_text(text):
    text = re.sub(r'<[^>]+>', '', text)  # Remove HTML tags
    text = text.lower()  # Lowercase
    text = ' '.join(word for word in text.split() if word not in stop_words)  # Remove stopwords
    return text

data['email_content'] = data['email_content'].apply(clean_text)

# Extract domains from URLs
def extract_domain(url):
    if pd.isna(url):
        return None
    extracted = extract(url)
    return f"{extracted.domain}.{extracted.suffix}"

data['domain'] = data['urls'].apply(extract_domain)

# CTI: Flag suspicious domains (example list from PhishTank)
suspicious_domains = ['badbank.com', 'fake-login.net']
data['is_suspicious_domain'] = data['domain'].isin(suspicious_domains).astype(int)

2. Exploratory Data Analysis (EDA)





Visualize the distribution of phishing vs. legitimate emails.



Analyze word frequency in phishing emails to identify common CTI indicators (e.g., "password reset," "account suspension").



Plot the prevalence of suspicious domains or sender patterns.

Code Example:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# Class distribution
sns.countplot(x='label', data=data)
plt.title('Phishing vs Legitimate Emails')
plt.savefig('class_distribution.png')

# Word cloud for phishing emails
phishing_text = ' '.join(data[data['label'] == 1]['email_content'])
wordcloud = WordCloud(width=800, height=400).generate(phishing_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.savefig('phishing_wordcloud.png')

3. Feature Engineering





Create TF-IDF vectors for email content using scikit-learn’s TfidfVectorizer.



Add CTI-based features:





Binary indicator for suspicious domains.



Count of phishing-related keywords (e.g., "urgent," "click here").



Encode categorical features like sender domain using one-hot encoding.

Code Example:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(data['email_content'])

# Combine with CTI features
X_cti = data[['is_suspicious_domain']].values
import scipy.sparse as sp
X = sp.hstack([X_tfidf, X_cti])

# Labels
y = data['label'].values

4. Model Training





Train baseline models: Logistic Regression and Random Forest.



Train a BERT model using Hugging Face’s Transformers for advanced NLP-based classification.



Split data into 80% training and 20% testing sets.

Code Example (Baseline Models):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Evaluation
def evaluate_model(y_true, y_pred, model_name):
    print(f"{model_name} Performance:")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_true, y_pred):.4f}")

evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_rf, "Random Forest")

Code Example (BERT):

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Tokenize data
def tokenize_data(texts, labels):
    encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128)
    return {'input_ids': encodings['input_ids'], 'attention_mask': encodings['attention_mask'], 'labels': labels}

# Prepare dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(data['email_content'], data['label'], test_size=0.2, random_state=42)
train_dataset = tokenize_data(train_texts, train_labels)
test_dataset = tokenize_data(test_texts, test_labels)

# Convert to PyTorch tensors
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings['labels'])

train_dataset = EmailDataset(train_dataset)
test_dataset = EmailDataset(test_dataset)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    logging_dir='./logs',
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train model
trainer.train()

# Evaluate
eval_results = trainer.evaluate()
print("BERT Evaluation:", eval_results)

5. CTI Integration





Enhance the model by incorporating CTI feeds:





Use a list of known phishing domains from PhishTank or OpenPhish.



Add MITRE ATT&CK tactics (e.g., T1566: Phishing) to flag emails with specific patterns (e.g., urgent calls to action).



Re-train the model with these additional features to improve precision.

Code Example:

In [ ]:
# Example CTI keyword list from MITRE ATT&CK phishing tactics
phishing_keywords = ['urgent', 'verify', 'login', 'password', 'account', 'click here']
data['keyword_count'] = data['email_content'].apply(lambda x: sum(1 for word in phishing_keywords if word in x.lower()))

# Update feature matrix
X_cti = data[['is_suspicious_domain', 'keyword_count']].values
X = sp.hstack([X_tfidf, X_cti])

# Re-train Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
evaluate_model(y_test, y_pred_rf, "Random Forest with CTI")

6. Model Evaluation





Compare model performance using:





Accuracy: Overall correctness.



Precision: Proportion of correctly identified phishing emails.



Recall: Proportion of actual phishing emails detected.



F1-Score: Balance between precision and recall.



Visualize results using a confusion matrix.

Code Example:

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png')

7. Deployment





Create a simple script to classify new emails in real-time.



Input: Email content and metadata.



Output: Probability of being a phishing email.

Code Example:

In [ ]:
def classify_email(email_content, url=None):
    # Preprocess
    cleaned_text = clean_text(email_content)
    tfidf_vector = tfidf.transform([cleaned_text])
    domain = extract_domain(url) if url else None
    is_suspicious = 1 if domain in suspicious_domains else 0
    keyword_count = sum(1 for word in phishing_keywords if word in cleaned_text.lower())
    
    # Combine features
    features = sp.hstack([tfidf_vector, [[is_suspicious, keyword_count]]])
    
    # Predict
    prob = rf.predict_proba(features)[0][1]
    return f"Phishing Probability: {prob:.2%}"

# Example usage
email = "Urgent: Verify your account now at http://badbank.com"
url = "http://badbank.com"
print(classify_email(email, url))

Expected Results





Baseline Models:





Logistic Regression: ~85% accuracy, ~80% F1-score.



Random Forest: ~90% accuracy, ~85% F1-score.



BERT Model: ~92% accuracy, ~90% F1-score (due to advanced NLP capabilities).



With CTI Features: Random Forest improves to ~92% accuracy, with higher precision (~90%) due to CTI-based domain and keyword flags.